# VietMedBridge — 02: Nhập bản crawl 100k → source + parent/child chunks

**Runtime CPU, Run all.** Đọc archive `vibiomir_shard_00000.tar` hoặc thư mục
đã giải nén, hoặc thư mục `.tar.parts` có archive_manifest.json.
Bản chia phần được tự ghép trên ổ local Colab và kiểm SHA-256 đúng bản gốc.
Tái sử dụng văn bản EXTRACT_SUCCESS; không tải lại các website.
Archive và snapshot BTC phải cùng SHA-256. Map bằng official URL, giữ toàn bộ
official ID/alias và một outcome cho từng ID. Nguồn lỗi/challenge/redirect về
trang chủ được ghi failures để xử lý sau. LOW quality được giữ kèm flags.

Mặc định tìm archive trong `VietMedBridge/data/incoming`,
`VietMedBridge/data/data_temp` hoặc `VietMedBridge/data_temp` trên Drive.
Đặt EXTERNAL_SOURCE nếu archive nằm nơi khác. Sau khi ghép (nếu cần), copy bốn file metadata
từ archive sang ổ local Colab để đọc; raw archive vẫn giữ làm nguồn kiểm tra.
Metadata binding và exact-source spans không chứng nhận chất lượng y khoa.

Ghi theo shard 2.048 ID, checkpoint trên Drive. Khi ngắt, Run all với cùng
đường dẫn input/BUILD_RUN/config để tiếp tục. Một runtime ghi một build.
Không tạo embedding, không cần GPU. Sau COMPLETE, chạy notebook 03.


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError(
        "Dùng Python 3.11–3.13. Trong Runtime > Change runtime type, "
        "chọn phiên bản runtime có Python trong khoảng này."
    )
if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Notebook này được thiết kế cho Google Colab.")

from google.colab import drive
# Chạy lại cell này sau MỖI runtime mới/restart: package chỉ được cài trong phiên Colab.
drive.mount("/content/drive")

# Giữ cùng DATA_ROOT trong cả bốn notebook.
DATA_ROOT = Path("/content/drive/MyDrive/VietMedBridge/data")
WORK_DIR = Path("/content/vmb_work")  # temp files/spill ở ổ local của Colab
DATA_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
CODE_REVISION = None  # Có thể đặt full commit SHA; mặc định dùng code_lock đã lưu.
EXPECTED_PIPELINE_API = 2
CHECKOUT = Path("/content/VietMedBridge")
lock_path = DATA_ROOT / "data_processing_code_lock.json"
lock = json.loads(lock_path.read_text()) if lock_path.exists() else {}
if lock and lock.get("repo_url") != REPO_URL:
    raise ValueError("DATA_ROOT đang khóa vào một repo khác.")
if lock and lock.get("pipeline_api") != EXPECTED_PIPELINE_API and not CODE_REVISION:
    raise RuntimeError(
        "DATA_ROOT đang dùng code cũ. Để nâng lên data-v2, đặt CODE_REVISION='main' "
        "ở cell này rồi chạy lại; dùng tên crawl/build run mới. Raw cũ vẫn được giữ."
    )
DATA_WORKFLOW_API = "external-extraction-import-v1"
upgrade = lock.get("workflow_api") != DATA_WORKFLOW_API
reference = CODE_REVISION or ("main" if upgrade else lock.get("git_commit")) or "main"
if not (CHECKOUT / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CHECKOUT)], check=True)
remote = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True
).strip()
if remote != REPO_URL:
    raise ValueError("Checkout hiện tại không thuộc repo VietMedBridge.")
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", reference], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "rev-parse", "FETCH_HEAD"], text=True
).strip()
if "vietmedbridge" in sys.modules and globals().get("_VMB_IMPORTED_COMMIT") != CODE_COMMIT:
    raise RuntimeError("Mã nguồn đổi trong runtime đã import package. Restart session rồi chạy lại notebook.")
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"],
    cwd=CHECKOUT,
    check=True,
)
PACKAGE_SOURCE = CHECKOUT / "src"
if not (PACKAGE_SOURCE / "vietmedbridge" / "__init__.py").is_file():
    raise RuntimeError(
        f"Repo checkout thiếu package source: {PACKAGE_SOURCE}; commit={CODE_COMMIT}. "
        "Kiểm tra lại git fetch/checkout ở output của cell Bootstrap."
    )
sys.path.insert(0, str(PACKAGE_SOURCE))
importlib.invalidate_caches()
from vietmedbridge.artifacts import atomic_json, runtime_versions
from vietmedbridge import PIPELINE_API_VERSION
if PIPELINE_API_VERSION != EXPECTED_PIPELINE_API:
    raise RuntimeError("Notebook và package API không khớp; chọn commit data-v2 và restart session.")
_VMB_IMPORTED_COMMIT = CODE_COMMIT
if not lock or CODE_REVISION or upgrade:
    atomic_json(lock_path, {"repo_url": REPO_URL, "git_commit": CODE_COMMIT, "pipeline_api": PIPELINE_API_VERSION, "workflow_api": DATA_WORKFLOW_API})
PIPELINE_CONFIG = json.loads((CHECKOUT / "configs/data_pipeline.json").read_text())
atomic_json(DATA_ROOT / "data_processing_runtime.json", {
    "git_commit": CODE_COMMIT, "python": sys.version, "packages": runtime_versions()
})
print("Code commit:", CODE_COMMIT)
import vietmedbridge
print("Package source:", Path(vietmedbridge.__file__).resolve())
print("Dữ liệu/checkpoint:", DATA_ROOT)


## 1. Chọn nguồn và tokenizer


In [ ]:
from vietmedbridge.artifacts import read_json, atomic_json
from vietmedbridge.dataset import load_snapshot, parquet_path
from vietmedbridge.chunks import ChunkConfig, load_bge_tokenizer
from vietmedbridge.external_import import find_external_source, import_external_corpus

INPUT_KIND = "external"  # dùng "crawl" cho các crawl run chuẩn cũ
EXTERNAL_SOURCE = None  # hoặc "/content/drive/MyDrive/.../vibiomir_shard_00000.tar"
BUILD_RUN = "team-100k-data-v1"
CRAWL_RUN = "stage-a-v2"  # chỉ dùng với INPUT_KIND="crawl"
SHARD_SIZE = 2048
MAX_NEW_SHARDS = None  # None = xử lý hết; giữ cùng config khi resume

SNAPSHOT = load_snapshot(DATA_ROOT)
OFFICIAL_LINKS = parquet_path(DATA_ROOT, "links_corpus.parquet")
tokenizer_lock_path = DATA_ROOT / "tokenizer_lock.json"
lock = read_json(tokenizer_lock_path) if tokenizer_lock_path.exists() else {}
TOKENIZER, TOKENIZER_SPEC = load_bge_tokenizer(lock.get("revision"))
if not lock:
    atomic_json(tokenizer_lock_path, TOKENIZER_SPEC)
CHUNK_CONFIG = ChunkConfig(**PIPELINE_CONFIG["chunking"])
if INPUT_KIND == "external":
    SOURCE = find_external_source(DATA_ROOT, EXTERNAL_SOURCE)
    if SOURCE is None:
        raise FileNotFoundError(
            "Chưa thấy archive 100k trên Drive. Đặt EXTERNAL_SOURCE bằng đường dẫn tar "
            "hoặc thư mục giải nén. Vị trí mặc định: "
            + str(DATA_ROOT / "incoming/vibiomir_shard_00000.tar"))
    print("Existing crawl:", SOURCE)
elif INPUT_KIND != "crawl":
    raise ValueError("INPUT_KIND phải là external hoặc crawl.")
print("Build:", BUILD_RUN, "| tokenizer:", TOKENIZER_SPEC)


## 2. Nhập/chia chunk theo shard — CPU


In [ ]:
if INPUT_KIND == "external":
    BUILD = import_external_corpus(
        SOURCE, OFFICIAL_LINKS, DATA_ROOT / "processed", TOKENIZER, TOKENIZER_SPEC,
        run_name=BUILD_RUN, config=CHUNK_CONFIG, shard_size=SHARD_SIZE,
        max_shards=MAX_NEW_SHARDS, work_dir=WORK_DIR)
else:
    from vietmedbridge.build import build_corpus
    BUILD = build_corpus(DATA_ROOT / "crawl" / CRAWL_RUN, DATA_ROOT / "processed",
        TOKENIZER, TOKENIZER_SPEC, run_name=BUILD_RUN, config=CHUNK_CONFIG,
        max_shards=MAX_NEW_SHARDS, official_links=OFFICIAL_LINKS, work_dir=WORK_DIR)
BUILD_DIR = DATA_ROOT / "processed" / BUILD_RUN
print(json.dumps({k: BUILD[k] for k in (
    "state", "counts", "selected_shards", "requested_input_records", "selected_range_complete"
)}, ensure_ascii=False, indent=2))
if BUILD["selected_range_complete"]:
    atomic_json(DATA_ROOT / "active_data_build.json", {
        "build_run": BUILD_RUN, "crawl_run": BUILD.get("crawl_run"),
        "signature": BUILD["signature"], "snapshot_sha256": BUILD["snapshot_sha256"],
        "input_kind": BUILD.get("input_kind", "CRAWL"), "state": BUILD["state"]})
    print("Hoàn tất input range. Chạy notebook 03 trên CPU.")
else:
    raise RuntimeError("Import chưa đủ range. Run all cùng config để tiếp tục trước notebook 03.")


## 3. Kiểm tra một tài liệu và thời gian thực đo


In [ ]:
import pyarrow.parquet as pq
from vietmedbridge.build import verify_spans
for part in BUILD["parts"]:
    path = BUILD_DIR / part["files"]["documents"]["path"]
    if pq.ParquetFile(path).metadata.num_rows == 0:
        continue
    doc = next(pq.ParquetFile(path).iter_batches(batch_size=1)).to_pylist()[0]
    children = pq.read_table(BUILD_DIR / part["files"]["children"]["path"], filters=[("doc_id", "=", doc["doc_id"])]).to_pylist()
    parents = pq.read_table(BUILD_DIR / part["files"]["parents"]["path"], filters=[("doc_id", "=", doc["doc_id"])]).to_pylist()
    verify_spans(doc, children, parents)
    print("Source:", doc["doc_id"], doc["title"], "| children:", len(children))
    print(children[0]["text"][:800] if children else "NO CHILDREN")
    break
timing = BUILD_DIR / "import_runtime.json"
if timing.exists():
    print(json.dumps(read_json(timing), ensure_ascii=False, indent=2))
print("Failures được bảo toàn trong các shard, cùng ID BTC.")


Outcome đủ range bao gồm nguồn lỗi, không có nghĩa 100% URL đã có nội dung tốt.
Notebook 03 kiểm toàn snapshot, freeze candidate và chuẩn bị các phần input
embedding trên disk. Notebook 04 sẽ đo tài nguyên trên mẫu nhỏ trước corpus lớn.
